In [0]:
# Create a watermark table to track last loaded timestamp
# This is our bookmark — we will update it every time we load new data

spark.sql("""
    CREATE TABLE IF NOT EXISTS watermark_table (
        table_name STRING,
        last_loaded_timestamp TIMESTAMP
    )
""")

# Check if watermark already exists for orders
watermark = spark.sql("""
    SELECT last_loaded_timestamp 
    FROM watermark_table 
    WHERE table_name = 'bronze_orders'
""")

# If no watermark exists yet, set it to beginning of time
if watermark.count() == 0:
    spark.sql("""
        INSERT INTO watermark_table VALUES 
        ('bronze_orders', '2000-01-01 00:00:00')
    """)
    print("Watermark created for first time!")
else:
    print("Existing watermark found!")
    watermark.show()

Existing watermark found!
+---------------------+
|last_loaded_timestamp|
+---------------------+
|  2018-10-17 17:30:18|
+---------------------+



In [0]:
# Read the watermark timestamp
# This tells us — "only load data AFTER this timestamp"

last_timestamp = spark.sql("""
    SELECT last_loaded_timestamp 
    FROM watermark_table 
    WHERE table_name = 'bronze_orders'
""").collect()[0][0]

print("Last loaded timestamp:", last_timestamp)
print("We will only load orders AFTER this time")

Last loaded timestamp: 2018-10-17 17:30:18
We will only load orders AFTER this time


In [0]:
# Read only NEW orders that came AFTER our last timestamp
# This is the core of incremental loading

from pyspark.sql.functions import col, to_timestamp

raw_orders = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/olist_raw_data/olist_orders_dataset.csv")

# Convert timestamp column to proper timestamp type
raw_orders = raw_orders.withColumn(
    "order_purchase_timestamp",
    to_timestamp(col("order_purchase_timestamp"))
)

# Filter only NEW rows — after our watermark
new_orders = raw_orders.filter(
    col("order_purchase_timestamp") > last_timestamp
)

print("Total rows in source:", raw_orders.count())
print("New rows to load:", new_orders.count())
print("Skipping already loaded rows:", 
      raw_orders.count() - new_orders.count())

Total rows in source: 99441
New rows to load: 0
Skipping already loaded rows: 99441


In [0]:
# MERGE new orders into bronze table
# This is professional incremental loading
# Update if order already exists, Insert if it's new

from delta.tables import DeltaTable

# First save new orders to a temporary view
new_orders.createOrReplaceTempView("new_orders_temp")

# Check if bronze_orders table exists already
if spark.catalog.tableExists("bronze_orders_incremental"):
    
    # Table exists — do a MERGE
    spark.sql("""
        MERGE INTO bronze_orders_incremental AS target
        USING new_orders_temp AS source
        ON target.order_id = source.order_id
        WHEN MATCHED THEN
            UPDATE SET *
        WHEN NOT MATCHED THEN
            INSERT *
    """)
    print("MERGE completed — existing rows updated, new rows inserted!")
    
else:
    # Table doesn't exist yet — create it fresh
    new_orders.write.format("delta") \
        .mode("overwrite") \
        .saveAsTable("bronze_orders_incremental")
    print("First load — table created fresh!")

# Show final count
final_count = spark.table("bronze_orders_incremental").count()
print("Total rows in bronze_orders_incremental:", final_count)

MERGE completed — existing rows updated, new rows inserted!
Total rows in bronze_orders_incremental: 99441


In [0]:
# Update watermark — fixed version
# Handles the case when no new data is found

from pyspark.sql.functions import max

if new_orders.count() > 0:
    # Only update watermark if we actually loaded new data
    latest_timestamp = new_orders.agg(
        max("order_purchase_timestamp")
    ).collect()[0][0]
    
    print("Latest timestamp in loaded data:", latest_timestamp)
    
    spark.sql(f"""
        UPDATE watermark_table
        SET last_loaded_timestamp = '{latest_timestamp}'
        WHERE table_name = 'bronze_orders'
    """)
    
    print("Watermark updated successfully!")
    print("Next run will only load orders after:", latest_timestamp)

else:
    print("No new data found — watermark stays the same!")
    print("Pipeline completed successfully with 0 new rows!")

No new data found — watermark stays the same!
Pipeline completed successfully with 0 new rows!


In [0]:
# TEST — Run incremental load again
# Since we already loaded everything, new rows should be 0

# Read watermark again
last_timestamp = spark.sql("""
    SELECT last_loaded_timestamp 
    FROM watermark_table 
    WHERE table_name = 'bronze_orders'
""").collect()[0][0]

print("Current watermark:", last_timestamp)

# Read source and filter
raw_orders = spark.read.format("csv") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load("/Volumes/workspace/default/olist_raw_data/olist_orders_dataset.csv")

raw_orders = raw_orders.withColumn(
    "order_purchase_timestamp",
    to_timestamp(col("order_purchase_timestamp"))
)

new_orders = raw_orders.filter(
    col("order_purchase_timestamp") > last_timestamp
)

print("New rows found:", new_orders.count())

if new_orders.count() == 0:
    print("No new data to load — incremental loading is working perfectly!")
else:
    print("New rows to load:", new_orders.count())

Current watermark: 2018-10-17 17:30:18
New rows found: 0
No new data to load — incremental loading is working perfectly!
